In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
from datetime import datetime
import pandas as pd

current_date = pd.to_datetime(datetime.utcnow()).floor('h')
print(f'{current_date}')

2026-06-26 07:00:00


In [3]:
from src.inference import load_batch_of_features_from_store

features = load_batch_of_features_from_store(current_date=current_date)

2026-06-26 10:55:40,497 INFO: Initializing external client
2026-06-26 10:55:40,501 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-26 10:55:42,398 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960
Fetching data from 2026-05-28 07:00:00 to 2026-06-26 06:00:00
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (12.84s) 
Rows per location:
count    259.0
mean     696.0
std        0.0
min      696.0
25%      696.0
50%      696.0
75%      696.0
max      696.0
dtype: float64


In [4]:
from src.inference import (
    load_model_from_registry,
    get_model_predictions
)

model = load_model_from_registry()
predictions = get_model_predictions(model, features)

2026-06-26 10:56:01,026 INFO: Closing external client and cleaning up certificates.
2026-06-26 10:56:01,035 INFO: Connection closed.
2026-06-26 10:56:01,037 INFO: Initializing external client
2026-06-26 10:56:01,037 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-26 10:56:02,498 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960
Using cached model files at 'd:\tmp\hopsworks\models\taxi_demand_prediction\taxi_demand_predictor_next_hour\2\taxi_demand_predictor_next_hour_2'. Pass local_path or call Model.clear_cache(...) to force a fresh download.
[LightGBM] [Warning] feature_fraction is set=0.2622501047068541, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.2622501047068541
[LightGBM] [Warning] bagging_fraction is set=0.9948789323285829, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9948789323285829


In [5]:
predictions['pickup_hour'] = current_date
predictions

,pickup_location_id,predicted_demand,pickup_hour
0,1,1.0,2026-06-26 07:00:00
1,2,0.0,2026-06-26 07:00:00
2,3,1.0,2026-06-26 07:00:00
3,4,16.0,2026-06-26 07:00:00
4,6,0.0,2026-06-26 07:00:00
...,...,...,...
254,261,9.0,2026-06-26 07:00:00
255,262,112.0,2026-06-26 07:00:00
256,263,88.0,2026-06-26 07:00:00
257,264,10.0,2026-06-26 07:00:00


Save these predictions in the feature store, so they can be later consumed by our `Streamlit app`

In [6]:
from src.feature_store_api import get_feature_store
from src import config

# connecto to the feature group
feature_group = get_feature_store().get_or_create_feature_group(
    name=config.FEATURE_GROUP_MODEL_PREDICTIONS,
    version=1,
    description="Predictions generated by our production model",
    primary_key=["pickup_location_id", "pickup_hour"],
    event_time="pickup_hour",
    time_travel_format="HUDI"
)

2026-06-26 10:56:06,597 INFO: Closing external client and cleaning up certificates.
2026-06-26 10:56:06,601 INFO: Connection closed.
2026-06-26 10:56:06,601 INFO: Initializing external client
2026-06-26 10:56:06,605 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-06-26 10:56:08,046 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/34960


In [7]:
feature_group.insert(predictions, write_options={"wait_for_job": True})

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/34960/fs/22624/fg/44311


Uploading Dataframe: 100.00% |██████████| Rows 259/259 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: model_predictions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/34960/jobs/named/model_predictions_1_offline_fg_materialization/executions


2026-06-26 10:57:22,362 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-06-26 10:57:28,694 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-06-26 10:59:38,535 INFO: Waiting for execution to finish. Current state: SUCCEEDING. Final status: UNDEFINED
2026-06-26 10:59:41,715 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: SUCCEEDED
2026-06-26 10:59:41,860 INFO: Waiting for log aggregation to finish.
2026-06-26 10:59:50,524 INFO: Execution finished successfully.


(Job('model_predictions_1_offline_fg_materialization', 'SPARK'), None)